### ReAct - Reason and Act

This pattern is mostly used for agentic systems where you need to decsribe what tools are required to do a particular job. So its just not the model anymore it also involves tools.

Within Dspy eco-system, react architecture pattern exists. Note that this is not the only the thing that is available as it relates to dspy.
It easily integrates with other optimizers which would also help tune the definitions of these tools itself.

So you get both

In [1]:
import dspy
import os
from dotenv import load_dotenv

load_dotenv()

ANTHROPIC_KEY = os.getenv("ANTHROPIC_KEY")
OPENAI_KEY = os.getenv("OPENAI_KEY")

In [2]:
lm = dspy.LM("anthropic/claude-opus-4-6", api_key=ANTHROPIC_KEY)
dspy.configure(lm=lm, cache=False)

### Example 1 - Answer questions about numbers

In [3]:
# Tool 1 - Simple eval + return result

def calculate(expression:str) -> str:
    """ Evaluates a matehmatical expression and returns the result.
    Args:
    expression(str) - A string that is expected to evaluate to a mathematical expression
    
    Returns:
    str : Returns the result if the string successfully evaluates to an expression otherwise error message is returned
    """
    try:
        result = eval(expression)
        return str(result)
    except Exception as e:
        return f"Error message {e}"


class ExpressionEval(dspy.Signature):
    """Answer the question using calculator tool"""

    expression:str = dspy.InputField(desc="Mathematical expression")
    result:str = dspy.OutputField(desc="A output result")


react = dspy.ReAct(signature=ExpressionEval, tools=[calculate])

print(react(expression="What is 25 percent of 200?").result)
        
    

25 percent of 200 is 50.


In [4]:
dspy.inspect_history(n=1)





[2026-04-10T18:51:36.695548]

System message:

Your input fields are:
1. `expression` (str): Mathematical expression
2. `trajectory` (str):
Your output fields are:
1. `reasoning` (str): 
2. `result` (str): A output result
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## expression ## ]]
{expression}

[[ ## trajectory ## ]]
{trajectory}

[[ ## reasoning ## ]]
{reasoning}

[[ ## result ## ]]
{result}

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        Answer the question using calculator tool


User message:

[[ ## expression ## ]]
What is 25 percent of 200?

[[ ## trajectory ## ]]
[[ ## thought_0 ## ]]
I need to calculate 25 percent of 200. This is equivalent to (25/100) * 200 or 0.25 * 200.

[[ ## tool_name_0 ## ]]
calculate

[[ ## tool_args_0 ## ]]
{"expression": "0.25 * 200"}

[[ ## observation_0 ## ]]
50.0

[[ ## thought_1 ## ]]
The calculation is complete. 25 percent of 200 is 50.0. I can now f

In [5]:
print(react(expression="25*&^23").result) ##Exception case

Error: invalid syntax. The expression "25*&^23" is not a valid mathematical expression due to the use of invalid characters (`&` and the combination `*&^`).


In [6]:
dspy.inspect_history(n=1)





[2026-04-10T18:51:36.807253]

System message:

Your input fields are:
1. `expression` (str): Mathematical expression
2. `trajectory` (str):
Your output fields are:
1. `reasoning` (str): 
2. `result` (str): A output result
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## expression ## ]]
{expression}

[[ ## trajectory ## ]]
{trajectory}

[[ ## reasoning ## ]]
{reasoning}

[[ ## result ## ]]
{result}

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        Answer the question using calculator tool


User message:

[[ ## expression ## ]]
25*&^23

[[ ## trajectory ## ]]
[[ ## thought_0 ## ]]
The expression "25*&^23" contains invalid characters (`&` and `^` in this context seems problematic). Let me try to evaluate it using the calculator tool to see what result or error message we get.

[[ ## tool_name_0 ## ]]
calculate

[[ ## tool_args_0 ## ]]
{"expression": "25*&^23"}

[[ ## observation_0 ## ]]
Error mess

#### Travel budget planner
Tools: convert currency, calculate per day budget, check if trip is affordable. 
    User asks things like "I have 2000 CAD, trip is 8 days, hotel is 80 USD per night, is this feasible?"

In [7]:
## Tools -  Will wrap up the exchange rates dictionary within a class just so that there is a structure to it

#Tool 1 - curreny converting
EXCHANGE_RATES = {
    "USD": 1.0,
    "CAD": 0.73,
    "EUR": 1.08,
    "GBP": 1.27,
    "JPY": 0.0067,
    "AUD": 0.65,
    "CHF": 1.13,
    "INR": 0.012,
    "MXN": 0.058,
    "SGD": 0.74,
}

class CurrencyConverter:
    def __init__(self):
        self.rates = EXCHANGE_RATES

    def convert_exchange_rates(self, amount:float, from_curr:str, to_curr:str) -> float | str:
        """This function converts a given amount from one currency type to another. Keep in mind that all values are 1 unit of this 
        currency equals X USD. So use USD as a baseline.
        """
        if (from_curr in self.rates.keys()) and (to_curr in self.rates.keys()):
            usd_amount = amount * self.rates[from_curr]
            return usd_amount / self.rates[to_curr]
        return "Exchange rate is not found for this current types"



##Tool 2  - calculate trip cost
def calculate_total_trip_cost(total_days:float) -> float|str:
    """ This function calculates total trip cost. Includes hotel, food and miscellaneous costs
    Args:
        total_days - Trip duration
    """
    hotel = 80
    food = 30
    misc = 40
    if total_days > 0:
        return (hotel+food+misc)  * total_days
    return "Trip duration should be greater than 0"

#Tool 3
def check_affordability(total_cost:float, total_budget:float) -> str:
    """ This function is used to check if the guest can afford this trip given their total budget.
        Use same currencies for this conversion
    """
    if total_cost > total_budget:
        return f"You can't afford this trip. Stay home. You are short by {abs(total_cost-total_budget)}"
    return f"You can afford this trip. Happy travels. Your surplus budget is {total_budget-total_cost}"



##Define signature
class TravelTripPlanner(dspy.Signature):
    """ You are a seasoned travel planner. Your job is to utilize the information and tools given to you to help a user to help them
    check if they have enough budget to cover expense. Assume the user is in United States unless specified
    In the process of travel planning you need to access tools that are available to you. Below is the descriptin
    1) convert_exchange_rates - Convert currency to target destination. 
    2) calculate_total_trip_cost - Helps you to calculate hotel, food, and miscellaneous expenses i.e. total_cost. Note that the results
    here are in USD. Convert it into right currency using convert_exchange_rates to do calculation.
    3) check_affordability - Use the total cost from calculate_total_trip_cost and total_budget from convert_exchange_rates to see if
    the user can afford this trip

    Ask followup questions if required. be respectful. DONOT ask for user preferences. You dont have that capability for now
    """

    ip:list = dspy.InputField(desc="Input Query")
    op:str = dspy.OutputField(desc="Response")
    

In [8]:
converter = CurrencyConverter()
trip_react = dspy.ReAct(signature=TravelTripPlanner, tools=[converter.convert_exchange_rates, calculate_total_trip_cost,
                                                           check_affordability,])

In [9]:
print(trip_react(ip=["""I am planning a trip to Singapore for 5 days with a budget of 3000$. Can you help me check if I can afford the trip 
and also tell me the total cost plus daily breakdown"""]).op)

Great news! 🎉 **You can absolutely afford your trip to Singapore!** Here's the full breakdown:

---

### 💰 Budget Overview
| | USD ($) | SGD (S$) |
|---|---|---|
| **Your Budget** | $3,000.00 | S$4,054.05 |
| **Total Trip Cost (5 days)** | $750.00 | S$1,013.51 |
| **Surplus** | $2,250.00 | S$3,040.54 |

---

### 📅 Daily Breakdown (per day)
| Expense Category | USD ($) | SGD (S$) |
|---|---|---|
| **Hotel, Food & Miscellaneous** | $150.00 | S$202.70 |

> **Daily cost: $150.00 USD (~S$202.70 SGD)**
> **Total cost for 5 days: $750.00 USD (~S$1,013.51 SGD)**

---

### ✅ Affordability Check
You are well within your budget! After covering all estimated expenses, you'll have a **surplus of approximately $2,250 USD (~S$3,040.54 SGD)**, which gives you plenty of room for shopping, attractions, or upgrading your experience in Singapore.

Happy travels! 🇸🇬✈️ If you need any further assistance, feel free to ask!


In [10]:
dspy.inspect_history(n=1)





[2026-04-10T18:51:37.029422]

System message:

Your input fields are:
1. `ip` (list): Input Query
2. `trajectory` (str):
Your output fields are:
1. `reasoning` (str): 
2. `op` (str): Response
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## ip ## ]]
{ip}

[[ ## trajectory ## ]]
{trajectory}

[[ ## reasoning ## ]]
{reasoning}

[[ ## op ## ]]
{op}

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        You are a seasoned travel planner. Your job is to utilize the information and tools given to you to help a user to help them
        check if they have enough budget to cover expense. Assume the user is in United States unless specified
        In the process of travel planning you need to access tools that are available to you. Below is the descriptin
        1) convert_exchange_rates - Convert currency to target destination. 
        2) calculate_total_trip_cost - Helps you to calculate hotel, food, and 

In [11]:
## Deficit case
print(trip_react(ip=["I am planning a trip to London for 10 days with a budget of 500 CAD. Can I afford it?"]).op)

Thank you for reaching out! I've looked into the costs for your 10-day trip to London. Here's a summary:

- **Estimated Total Trip Cost:** $1,500.00 USD
- **Your Budget:** 500 CAD (approximately **$365.00 USD**)

Unfortunately, **your current budget would not be sufficient** to cover this trip. You would be short by approximately **$1,135.00 USD** (roughly **1,554 CAD**).

To comfortably afford this 10-day London trip, you'd need a budget of around **2,055 CAD** or more.

Here are a couple of suggestions:
- **Shorten your trip** — a shorter stay would bring costs down.
- **Save up a bit more** before booking to ensure you have enough to enjoy your time in London without financial stress.

I hope this helps with your planning! Let me know if you'd like me to look into anything else. 😊


In [12]:
dspy.inspect_history(n=1)





[2026-04-10T18:51:37.216883]

System message:

Your input fields are:
1. `ip` (list): Input Query
2. `trajectory` (str):
Your output fields are:
1. `reasoning` (str): 
2. `op` (str): Response
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## ip ## ]]
{ip}

[[ ## trajectory ## ]]
{trajectory}

[[ ## reasoning ## ]]
{reasoning}

[[ ## op ## ]]
{op}

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        You are a seasoned travel planner. Your job is to utilize the information and tools given to you to help a user to help them
        check if they have enough budget to cover expense. Assume the user is in United States unless specified
        In the process of travel planning you need to access tools that are available to you. Below is the descriptin
        1) convert_exchange_rates - Convert currency to target destination. 
        2) calculate_total_trip_cost - Helps you to calculate hotel, food, and 

In [13]:
## Give less informatipn
messages = list()
while True:
    query = input("**Query: ").strip()
    if query.lower() == "exit" or query.lower() == "quit":
        break
    messages.append({"role" : "user",
                    "content":query})
    res = trip_react(ip=messages)
    print(f"\nSystem response: {res.op}")
    messages.append({"role" : "system",
                    "content":res})
    

**Query:  i wanr to got to australia



System response: That sounds like a wonderful destination! Australia has so much to offer. 🌏

To help you check if your budget covers the trip, I'll need a couple of details:

1. **What is your total budget for the trip?** (And which currency is it in?)
2. **How many days are you planning to stay in Australia?**

Once I have these, I can estimate your travel costs and let you know if it fits within your budget!


**Query:  2000usd for 10 days



System response: Great news! 🎉 You **can afford** your 10-day trip to Australia! Here's the breakdown:

| Detail | USD | AUD |
|---|---|---|
| **Your Budget** | $2,000.00 | ~$3,076.92 |
| **Estimated Trip Cost** (hotel, food & misc.) | $1,500.00 | ~$2,307.69 |
| **Surplus** | **$500.00** | **~$769.23** |

You'll have roughly **$500 USD (~769 AUD)** left over, which you could put toward activities, souvenirs, or any unexpected expenses.

A few things to keep in mind:
- This estimate covers **hotel, food, and miscellaneous expenses**. Make sure to also budget for **flights** if they aren't already accounted for, as they can be a significant cost.
- Australia can vary in cost depending on the city — Sydney and Melbourne tend to be pricier than other areas.

Have a fantastic trip to Australia! 🇦🇺✈️


**Query:  exit


In [14]:
dspy.inspect_history(n=1)





[2026-04-10T18:52:35.607052]

System message:

Your input fields are:
1. `ip` (list): Input Query
2. `trajectory` (str):
Your output fields are:
1. `reasoning` (str): 
2. `op` (str): Response
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## ip ## ]]
{ip}

[[ ## trajectory ## ]]
{trajectory}

[[ ## reasoning ## ]]
{reasoning}

[[ ## op ## ]]
{op}

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        You are a seasoned travel planner. Your job is to utilize the information and tools given to you to help a user to help them
        check if they have enough budget to cover expense. Assume the user is in United States unless specified
        In the process of travel planning you need to access tools that are available to you. Below is the descriptin
        1) convert_exchange_rates - Convert currency to target destination. 
        2) calculate_total_trip_cost - Helps you to calculate hotel, food, and 